# Session 1. An agent is a loop

**A working agent by the end. No framework in it.**

- a `for` statement, a text parser, one plain Python function
- the course contract: separate document


## The model as a component

**One function. Messages in, one message back.**

- `system`, `user`, `assistant`: the whole API surface today
- no wrapper today: the provider's own client, nothing hidden
- `MODEL_CHEAP` today, `MODEL_STRONG` from session 2, both from `.env`
- two named model families shut down this semester, one in October near session 6


In [ ]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # reads .env from here or any directory above

API_KEY = os.environ["LLM_API_KEY"]
BASE_URL = os.environ["LLM_BASE_URL"]  # wrong tail: a 404 that looks like auth
MODEL = os.environ["MODEL_CHEAP"]      # models die mid-term, edit .env not code

# gpt-5.x on chat completions: tools work only with reasoning switched off
EXTRA = {"reasoning_effort": os.environ["LLM_REASONING_EFFORT"]} if os.getenv("LLM_REASONING_EFFORT") else {}
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
print("endpoint:", BASE_URL, "| model:", MODEL)


In [ ]:
answer = client.chat.completions.create(
    model=MODEL,
    messages=[  # the whole conversation state, resent on every call
        {"role": "system", "content": "You are terse. Two sentences at most."},  # standing
        {"role": "user", "content": "Name one book about graph algorithms."},  # what you typed
    ],
    max_completion_tokens=2000,  # hidden reasoning is billed from this too
)

print(answer.choices[0].message)  # choices is a list; message holds tool_calls too
print()
print("finish_reason:", answer.choices[0].finish_reason)  # stop, or out of budget
print("usage:        ", answer.usage)


In [ ]:
def ask(messages):
    reply = client.chat.completions.create(
        model=MODEL, messages=messages, max_completion_tokens=2000
    )
    return reply.choices[0].message.content  # text only, the object came back above


Q = "Name one book about graph algorithms."

# identical answers here would be determinism, not memory
print("call 1:", ask([{"role": "user", "content": Q}]))
print()
print("call 2:", ask([{"role": "user", "content": Q}]))
print()
print("call 3:", ask([{"role": "user", "content": "Who wrote it?"}]))


**Three requests, three strangers.**

- call 3 asks who wrote *it*: there is no *it*, it answers anyway


In [ ]:
first = ask([{"role": "user", "content": Q}])  # one real call, then we fake history

conversation = [
    {"role": "user", "content": Q},
    {"role": "assistant", "content": first},
    {"role": "user", "content": "Who wrote it?"},
]

print(ask(conversation))  # call 3 again, and *it* now has an antecedent


**The conversation is a list you own and resend.**

- the provider stores nothing between calls
- memory bugs: the wrong list, an overwritten list, a copy
- sessions 2-16 add machinery, none replaces the list
- a black box with an interface; we teach the interface
- the box itself: https://karpathy.ai/zero-to-hero.html


## Access channels

**Table is in the pre-course document. One rule out loud.**

- free tier: assume the provider keeps what you send and that people read it
- tool output goes there too, so fictional data only


## The loop

**The agent needs something to call.**

- no decorator, no base class, plain Python
- deliberately dumb substring match, so it can miss


In [ ]:
BOOKS = [
    ("Introduction to Algorithms", "algorithms, data structures, graphs"),
    ("The Pragmatic Programmer", "craft, habits, career"),
    ("Designing Data-Intensive Applications", "databases, distributed systems"),
    ("Structure and Interpretation of Computer Programs", "recursion, interpreters"),
    ("Refactoring", "code smells, tests, design"),
    ("The Mythical Man-Month", "teams, estimation, schedules"),
]


def find_book(query: str) -> str:  # nothing here knows a model exists
    """Search the catalogue by title or topic. One short word works best."""
    needle = query.strip().lower()
    hits = [t for t, topics in BOOKS if needle in t.lower() or needle in topics]
    if not hits:
        # for the model, not the log: return None and it gives up
        return f"No match for {query!r}. Try a single keyword, for example: graphs, tests, databases."
    return "; ".join(hits)


print(find_book("graphs"))
print(find_book("recursion"))
print(find_book("something about graph algorithms"))  # the line the model hits first


**The failure message is the model's only input to its next move.**

- session 3 is largely about this one sentence


### The system prompt

**Read it out loud: this text is the whole protocol.**

- a shape agreed in advance, so `str.startswith` can find it


In [ ]:
# a model emits text, so tool calls are text
SYSTEM = """You answer questions using a book catalogue.

Work in a loop and use exactly this format:

Thought: what you need to find out
Action: find_book("<query>")
Observation: <what the tool returned>

When you know the answer, reply with:

Final Answer: <your answer>
"""

# safety one lives in the prompt; safety two is the cut below
STOP_RULE = "\nAfter the Action line STOP and wait: never write the Observation yourself."

# a phrase, not a keyword: the first call misses
QUESTION = "Can you recommend me something about graph algorithms?"

print(SYSTEM)


### Cutting the reply off

**"STOP and wait" is a request, not a mechanism.**

- current models obey that line; weaker ones write the Observation themselves
- the cut below holds on every endpoint in the pre-course table, rule or no rule
- cleverer parsers exist and fail in more interesting ways


In [ ]:
# stop parameter 400s on reasoning models: we cut here
def cut_at_observation(text: str) -> str:
    marker = text.find("Observation:")
    return text if marker == -1 else text[:marker].rstrip()  # -1: nothing to cut


# an uncut reply: the model answered its own tool call
FAKE = """Thought: I should look this up.
Action: find_book("graphs")
Observation: Introduction to Algorithms, third edition.
Final Answer: Read Introduction to Algorithms."""

print(cut_at_observation(FAKE))


In [ ]:
def parse_action(text: str) -> str | None:
    """Pull the query out of: Action: find_book("graphs")"""
    for line in text.splitlines():
        line = line.strip()
        if line.startswith("Action:") and "(" in line:  # take what is in brackets
            return line[line.index("(") + 1 : line.rindex(")")].strip().strip("\"'")
    return None  # done: the loop's only exit signal


print(repr(parse_action(FAKE)))
print(repr(parse_action("Final Answer: Introduction to Algorithms.")))


### Handing the result back

**The observation goes back as a `user` message.**

- the API will not continue a trailing assistant message
- Hugging Face appends it to the assistant turn: does not port
- six moves: send, cut, find Action, run, append two, send again
- one of the six belongs to the model. The rest is our code


In [ ]:
def run(honest: bool = True, steps: int = 5) -> list[str]:
    messages = [  # honest=False drops both safeties: the rule and the cut
        {"role": "system", "content": SYSTEM + (STOP_RULE if honest else "")},
        {"role": "user", "content": QUESTION},
    ]
    transcript = [f"Question: {QUESTION}"]  # the run log you will commit

    for _ in range(steps):
        answer = client.chat.completions.create(
            model=MODEL,
            messages=messages,  # resent in full: this list is the only memory
            max_completion_tokens=2000,
        )
        reply = answer.choices[0].message.content or ""  # content can be None, not ""
        if honest:
            reply = cut_at_observation(reply)
        transcript.append(reply)
        print(reply)

        query = parse_action(reply)
        if query is None:  # no Action line: the model is finished, or thinks so
            break

        observation = f"Observation: {find_book(query)}"  # our Python actually runs
        transcript.append(observation)
        print(observation)

        messages.append({"role": "assistant", "content": reply})
        messages.append({"role": "user", "content": observation})  # user, not assistant

    return transcript


**Bound every loop you write this semester.**

- session 2 calls it `recursion_limit`: langgraph 1.2 defaults to 10007, not 25


In [ ]:
transcript = run()  # count the Action lines: there should be two


**Two Action lines. The second one is the whole course.**

- the miss came back with advice, the model narrowed to one word, then answered
- its output changed what we sent next: that is all an agent is

### Now take both safeties off


In [ ]:
broken = run(honest=False)  # no STOP rule in the prompt, no cut in the code


**No exception, no warning. Right shape, invented world.**

- the model wrote its own Observation and a Final Answer before `find_book` ran
- the real Observation lands after that answer; the model just answers again
- correct by luck is worse, that is the version you ship
- the cut is what holds on every endpoint in the table, rule or no rule


### Two failures that get blamed on the parser

**Both come from the token budget. Neither is the parser's fault.**

- sixteen tokens: a reasoning model spends them on hidden thinking and returns nothing
- a plain model stops mid-Thought; `finish_reason` says `length` either way
- some days OpenAI returns the starved request as a 400 instead: same cause, third shape

In [ ]:
from openai import BadRequestError

try:
    tiny = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": QUESTION},
        ],
        max_completion_tokens=16,  # starved on purpose: hidden thinking is billed from this too
    )
    print("content:      ", repr(tiny.choices[0].message.content))
    print("finish_reason:", tiny.choices[0].finish_reason)  # length, and nothing raised
    print("usage:        ", tiny.usage)
except BadRequestError as error:  # same starvation, other shape: some days it is a 400
    print("400 from the provider:", error.body["error"]["message"])

In [ ]:
truncated = "Thought: I should search the cat"

print("content:      ", repr(truncated))
print("parse_action: ", repr(parse_action(truncated)))  # None, like a finished answer


**The parser is correct. The budget was wrong.**

- both parse to None, and None means finished
- an empty or half-written reply ships as the answer
- `agent/assistant.py` raises a named error instead


## The same request, with a schema

**One parameter. Watch what stops being our problem.**

- one model request only: a second turn 400s on Gemini's compat endpoint
- its reasoning signature does not survive the round trip; session 2 fixes it


In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "find_book",  # the name the model prints back to us
            # description and parameter docs are prompt, not documentation
            "description": "Search the book catalogue. One short keyword works best.",
            "parameters": {  # JSON Schema, carrying the format half of SYSTEM
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "A single keyword"}
                },
                "required": ["query"],
            },
        },
    }
]


In [ ]:
import json

native = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": QUESTION}],  # no system prompt at all
    tools=TOOLS,  # the only addition to the very first request
    max_completion_tokens=2000,
    **EXTRA,  # empty for most providers; see the setup cell
)
message = native.choices[0].message

print("content:   ", repr(message.content))
print("tool_calls:", message.tool_calls)

# one request only: turn two 400s on Gemini's compat endpoint
for call in message.tool_calls or []:
    arguments = json.loads(call.function.arguments)
    print(f"\ncalling {call.function.name}({arguments}) ->")
    print(find_book(**arguments))


**The interface did not gain an ability. It gained a schema.**

- the call arrives as an object; `content` is empty or a one-line remark, by provider
- arguments still need `json.loads`: it was a string on the wire
- gone from our code: the format prompt, and `parse_action`
- still ours: running the function
- feeding the result back is the loop we wrote by hand at the top of this notebook; session 2 stops writing it

## Practice

**Your own repository, from scratch.**

- `git init`, a `.env` with the four variables, one Python file
- four TODOs in order, the file runs after each
- your `parse_action` must find the format your prompt fixes
- you extend this file every session until December

```python
import os
import sys

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.environ["LLM_API_KEY"], base_url=os.environ["LLM_BASE_URL"])
MODEL = os.environ["MODEL_CHEAP"]

# TODO 1. Tool name, call format; the STOP rule stays separate
SYSTEM = """TODO"""
STOP_RULE = "\nAfter the Action line STOP and wait: never write the Observation yourself."
QUESTION = "TODO"


def my_tool(argument: str) -> str:
    """TODO 2. One string in, one string out. Rename it to what it does and keep
    TOOL pointing at it. The failure case must say what a better argument would
    look like: that sentence is all the model has to work with."""
    raise NotImplementedError


TOOL = my_tool


def parse_action(text: str) -> str | None:
    """TODO 3. Find the call, return the argument, return None when the model
    has finished. Line by line with str.startswith is enough."""
    raise NotImplementedError


def cut_at_observation(text: str) -> str:
    marker = text.find("Observation:")
    return text if marker == -1 else text[:marker].rstrip()


def run(honest: bool = True, steps: int = 5) -> list[str]:
    messages = [{"role": "system", "content": SYSTEM + (STOP_RULE if honest else "")},
                {"role": "user", "content": QUESTION}]
    transcript = [f"Question: {QUESTION}"]
    for _ in range(steps):
        answer = client.chat.completions.create(
            # pre-set high: hidden reasoning is billed from this budget too
            model=MODEL, messages=messages, max_completion_tokens=2000
        )
        choice = answer.choices[0]
        reply = choice.message.content or ""
        # a budget bug, not a parser bug: name it
        if not reply and choice.finish_reason == "length":
            raise RuntimeError(
                "Empty reply at the length limit: the budget went on hidden "
                "reasoning, not on your parser. Raise max_completion_tokens."
            )
        if honest:
            reply = cut_at_observation(reply)
        transcript.append(reply)
        print(reply)

        argument = parse_action(reply)
        if argument is None:
            break
        observation = f"Observation: {TOOL(argument)}"
        transcript.append(observation)
        print(observation)
        messages.append({"role": "assistant", "content": reply})
        # TODO 4. Hand the observation back: one line. Whose turn?
    return transcript


if __name__ == "__main__":
    # --break removes both safeties: the second transcript you commit
    run(honest="--break" not in sys.argv)
```


**Two rules for the tool.**

- pure Python, no network, no keys: debug your loop, not an API
- a domain no course demo uses: book search, plant care, currency rates, support tickets are taken
- examples list is in the pre-course document


**Required artifact: both transcripts, committed.**

- run it twice, plain and with `--break`
- both into `runs/session-01.md`, tool name and signature on top
- put both safeties back before you commit
- one sentence of your own on what the second run shows


**Stretch, if you finish early.**

- a real tool, not a lookup table, with a miss case worth reading
- session 5 is project choice: better to like what you wrote


## Today, in one card

**An agent is a loop whose next input is its own last output.**

**You can now defend:**
- the conversation is a list you own and resend: the provider stores nothing between calls
- a tool call is text; the schema moved the parsing to the provider, running the function stayed ours
- "STOP and wait" is a request the model may ignore; the cut in your code is the mechanism

**In your repository:** `runs/session-01.md`, two transcripts of your own loop, plain and with `--break`, one sentence on the difference.
**The trap of the day:** a starved token budget looks like a parser bug; `finish_reason` and `usage` say otherwise.
**Ask yourself:** why does the observation go back as a `user` message, and what breaks if it goes back as `assistant`?
**Next time:** the loop becomes a graph: `parse_action` disappears, and the observation becomes a tool message with an id.